# LSTM Cache Action Predictor

Driver notebook for `formal_NN_training`. The model is not only an SPP filter: it learns memory/cache actions including prefetch timing and bypass/low-priority insertion.

Use your GitHub pull/push setup cell in Colab, then run the cells below after the repo is cloned.


## Generate SPP trace data

Requires `external/ChampSim/` and `traces/`. Produces `formal_NN_training/data/generated/lstm_events_<TRACE>.csv`.


In [ ]:
import os, subprocess
os.environ.setdefault('TRACE','602.gcc_s-734B')
os.environ.setdefault('WARMUP','25000000')
os.environ.setdefault('SIM','25000000')
os.environ.setdefault('BUILD','1')
os.environ.setdefault('PATCH_SPP','1')
subprocess.run(['bash','formal_NN_training/scripts/01_run_spp_trace_dump.sh'], check=True)


## Train / export

Train your LSTM on CSVs under `formal_NN_training/data/generated/`. The expected exported action table is `formal_NN_training/artifacts/full_lstm_cache_actions.csv` or `val_lstm_cache_actions.csv`.


## Replay exported actions

This converts the action table to a `list_replayer` prefetch list and runs the existing ChampSim replay script.


In [ ]:
import os, subprocess
os.environ.setdefault('TRACE','602.gcc_s-734B')
os.environ.setdefault('WARMUP','25000000')
os.environ.setdefault('SIM','25000000')
os.environ.setdefault('PREFETCH_THRESHOLD','0.50')
os.environ.setdefault('BYPASS_THRESHOLD','0.60')
subprocess.run(['bash','formal_NN_training/scripts/03_run_lstm_replay.sh'], check=True)


## Push results

Stage and commit only generated artifacts/results. Set your authenticated remote in Colab before running `git push`.


In [ ]:
import subprocess, datetime
for p in ['formal_NN_training/artifacts','formal_NN_training/data/generated','formal_NN_training/results','results/generated/prefetch_lists','results/nn_demo_summary.csv']:
    subprocess.run(['bash','-lc',f'[ -e {p} ] && git add {p} || true'], check=True)
subprocess.run(['git','status','--short'], check=True)
if subprocess.run(['git','diff','--cached','--quiet']).returncode != 0:
    msg='Add LSTM cache-action results '+datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    subprocess.run(['git','commit','-m',msg], check=True)
    print('Committed:', msg)
else:
    print('Nothing staged')
